In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

import torch
import torch_geometric as pyg

from baseline_evals.feature_selection import variance_filtering

from torch_geometric.utils import to_networkx


from bipartite_gnn.train_test import GNNTrainer
from bipartite_gnn.model import GAT_2L, BipartiteRGAT
from bipartite_gnn.graph_building import cosine_similarity_matrix, dense_to_coo

In [3]:
null_vals = ["NA"]
mrna = pl.read_csv("BRCA_PROCESSED_DATA/mrna.tsv", separator="\t", null_values=null_vals)
cna = pl.read_csv("BRCA_PROCESSED_DATA/cnvth.tsv", separator="\t", null_values=null_vals)
mirna = pl.read_csv("BRCA_PROCESSED_DATA/mirna.tsv", separator="\t", null_values=null_vals)

labels = pl.read_csv("BRCA_PROCESSED_DATA/labels.tsv", separator="\t")
le = LabelEncoder()
le.fit(labels["PAM50_mRNA_nature2012"].to_list())
y = le.transform(labels["PAM50_mRNA_nature2012"].to_list())
# labels, y

In [4]:
# mrna, cna, mirna
mrna_gene_names = mrna[:, 0].to_list()
cna_gene_names = cna[:, 0].to_list()
mirna_gene_names = mirna[:, 0].to_list()

In [5]:
mrna_X = torch.tensor(mrna[:, 1:].to_numpy().T)
cna_X = torch.tensor(cna[:, 1:].to_numpy().T)
mirna_X = torch.tensor(mirna[:, 1:].to_numpy().T)

In [6]:
mrna_X.shape

torch.Size([483, 18975])

In [7]:
train_mask, test_mask = train_test_split(torch.arange(len(y)), test_size=0.3, stratify=y, random_state=3)
val_mask, test_mask = train_test_split(test_mask, test_size=0.5, random_state=3)

In [9]:
# select 400 top genes for mrna
select_mask = variance_filtering(mrna_X.numpy(), 400)
mrna_X_400 = mrna_X[:, select_mask]

# normalize mrna_X_400
mm_scaler = MinMaxScaler()
mm_scaler.fit(mrna_X_400.numpy()[train_mask])

mrna_X_400 = torch.tensor(mm_scaler.transform(mrna_X_400.numpy()))

# select 400 top genes for cna
select_mask = variance_filtering(cna_X.numpy(), 400)
cna_X_400 = cna_X[:, select_mask].float()

# select 200 top genes for mirna
select_mask = variance_filtering(mirna_X.numpy(), 200)
mirna_X_200 = mirna_X[:, select_mask]

In [42]:
# create hetero-data object
from bipartite_gnn.graph_building import create_diff_exp_connections_nbnom, create_diff_exp_connections_norm, dense_to_coo
import torch_geometric.transforms as T

data = pyg.data.HeteroData()

proj_dim = 128

mm = MinMaxScaler()
mm.fit(mrna_X_400.numpy()[train_mask])
mrna_X_400_norm = mm.transform(mrna_X_400.numpy())

# sample node features
data["mrna"].x = torch.tensor(mrna_X_400_norm).float()
# data["cna"].x = cna_X_400.float()
# data["mirna"].x = mirna_X_200.float()

data.y = torch.tensor(y)
# data["mrna"].y = data["cna"].y = data["mirna"].y = torch.tensor(y)

data.omics = ["mrna"] #, "cna", "mirna"]

# feature node projection
data["mrna_feature"].x = torch.zeros(mrna_X_400.shape[1], proj_dim)
# data["cna_feature"].x = torch.ones(cna_X_400.shape[1], proj_dim)
# data["mirna_feature"].x = torch.ones(mirna_X_200.shape[1], proj_dim)

# create edges
 # 3 for forward egdes, 3 for backward edges

mrna_A = create_diff_exp_connections_norm(mrna_X_400, 2.0)

print(mrna_A.shape)

print("isolated samples, isolated genes")
print((mrna_A.abs().sum(axis=1) == 0).sum(), (mrna_A.abs().sum(axis=0) == 0).sum())

data["mrna", "diff_exp", "mrna_feature"].edge_index = dense_to_coo(mrna_A)
data["mrna", "self_loop", "mrna"].edge_index = dense_to_coo(torch.eye(mrna_X_400.shape[0]))
data["mrna_feature", "self_loop", "mrna_feature"].edge_index = dense_to_coo(torch.eye(mrna_X_400.shape[1]))

# # data["cna", "diff_exp", "cna_feature"].edge_index = dense_to_coo(create_diff_exp_connections_norm(cna_X_400, 2.0))
# # data["mirna", "diff_exp", "mirna_feature"].edge_index = dense_to_coo(create_diff_exp_connections_norm(mirna_X_200, 2.0))

data = T.ToUndirected()(data)
# data = T.AddSelfLoops()(data)

# 2 for forward and backward diff exp, one for each self loop
data.num_relations = len(data.edge_index_dict.keys())
print("num_relations", data.num_relations)

data["train_mask"] = train_mask
data["val_mask"] = val_mask
data["test_mask"] = test_mask

data

torch.Size([483, 400])
isolated samples, isolated genes
tensor(14) tensor(1)
num_relations 4


HeteroData(
  y=[483],
  omics=[1],
  num_relations=4,
  train_mask=[386],
  val_mask=[48],
  test_mask=[49],
  mrna={ x=[483, 400] },
  mrna_feature={ x=[400, 128] },
  (mrna, diff_exp, mrna_feature)={ edge_index=[2, 6129] },
  (mrna, self_loop, mrna)={ edge_index=[2, 483] },
  (mrna_feature, self_loop, mrna_feature)={ edge_index=[2, 400] },
  (mrna_feature, rev_diff_exp, mrna)={ edge_index=[2, 6129] }
)

In [170]:
data.to_homogeneous()

Data(y=[483], omics=[1], num_relations=2, train_mask=[386], val_mask=[48], test_mask=[49], edge_index=[2, 17808], x=[883, 400], node_type=[883], edge_type=[17808])

In [43]:
model = BipartiteRGAT(
    input_sizes=[mrna_X_400.shape[1]], # , cna_X_400.shape[1], mirna_X_200.shape[1]],
    proj_dim=proj_dim,
    # proj_dropout=0.5,
    hidden_channels=[proj_dim, 64, 64], # num_layers = len(hidden_channels) - 1
    num_labels=len(np.unique(y)),
    num_relations=data.num_relations,
    num_bases= data.num_relations,
    num_heads=1,
    dropout=0.0,
    feature_integration_mode="linear",
)

trainer = GNNTrainer(
    model=model,
    loss_fn=torch.nn.CrossEntropyLoss(),
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4),
    params={
        "l1_lambda" : 0.001,
    }
)

trainer.train(
    data=data,
    epochs=20,
    log_interval=5,
)

x shape after projection: torch.Size([483, 128])
integration shape torch.Size([1, 483, 64])
x shape after projection: torch.Size([483, 128])
integration shape torch.Size([1, 483, 64])
x shape after projection: torch.Size([483, 128])
integration shape torch.Size([1, 483, 64])
x shape after projection: torch.Size([483, 128])
integration shape torch.Size([1, 483, 64])
x shape after projection: torch.Size([483, 128])
integration shape torch.Size([1, 483, 64])
x shape after projection: torch.Size([483, 128])
integration shape torch.Size([1, 483, 64])
x shape after projection: torch.Size([483, 128])
integration shape torch.Size([1, 483, 64])
Epoch: 005, 
Train Loss: 2.4732, Train Acc: 0.4326, Train F1 Macro: 0.1598, Train F1 Weighted: 0.2793
Val Loss: 1.3251, Val Acc: 0.5000, Val F1 Macro: 0.1667, Val F1 Weighted: 0.3472
Test Loss: 1.3282, Test Acc: 0.3673, Test F1 Macro: 0.1343, Test F1 Weighted: 0.2083
##################################################
x shape after projection: torch.Size(

In [ ]:
model(data.clone()).shape

Data(omics=[3], num_relations=6, edge_index=[2, 13842], x=[2449, 128], y=[2449], node_type=[2449], edge_type=[13842])
torch.Size([3, 483, 64])


torch.Size([483])

In [11]:
dh = data.to_homogeneous()

HeteroData(
  num_relations=6,
  mrna={
    x=[483, 400],
    y=[483],
  },
  cna={
    x=[483, 400],
    y=[483],
  },
  mirna={
    x=[483, 200],
    y=[483],
  },
  mrna_feature={ x=[18975, 128] },
  cna_feature={ x=[24776, 128] },
  mirna_feature={ x=[231, 128] },
  (mrna, diff_exp, mrna_feature)={ edge_index=[2, 3730] },
  (cna, diff_exp, cna_feature)={ edge_index=[2, 1997] },
  (mirna, diff_exp, mirna_feature)={ edge_index=[2, 1194] },
  (mrna_feature, rev_diff_exp, mrna)={ edge_index=[2, 3730] },
  (cna_feature, rev_diff_exp, cna)={ edge_index=[2, 1997] },
  (mirna_feature, rev_diff_exp, mirna)={ edge_index=[2, 1194] }
)

In [12]:
dh

Data(num_relations=6, edge_index=[2, 13842], x=[45431, 400], y=[45431], node_type=[45431], edge_type=[13842])

In [47]:
data

HeteroData(
  mrna={
    x=[483, 18975],
    y=[483],
  },
  cna={
    x=[483, 24776],
    y=[483],
  },
  mirna={
    x=[483, 231],
    y=[483],
  },
  mrna_feature={ x=[18975, 128] },
  cna_feature={ x=[24776, 128] },
  mirna_feature={ x=[231, 128] }
)

In [48]:
mrna_X.shape, cna_X.shape, mirna_X.shape

(torch.Size([483, 18975]), torch.Size([483, 24776]), torch.Size([483, 231]))

In [49]:
dh

Data(x=[45431, 24776], y=[45431], node_type=[45431])

In [50]:
# get counts of unique values in the dh.node_type tensor
node_type_counts = torch.unique(dh.node_type, return_counts=True)
node_type_counts

(tensor([0, 1, 2, 3, 4, 5]),
 tensor([  483,   483,   483, 18975, 24776,   231]))

In [19]:
data

HeteroData(
  mrna={
    x=[18975, 483],
    y=[483],
  },
  cna={
    x=[24776, 483],
    y=[483],
  },
  mirna={
    x=[231, 483],
    y=[483],
  }
)

In [ ]:
# train the model